In [ ]:
%load_ext autoreload
%autoreload 2

import os, glob
import numpy as np
import torch
import torch.nn as nn
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split
from scipy.io import wavfile
from PythonFiles.constants import Constants as C
from pathlib import Path

from PythonFiles.parsers import parse_phonemes, wav_to_logmel, windows_and_labels
from PythonFiles.parsers import PhonemeWindowDataset
from PythonFiles.NeuralModel import CRNN
from PythonFiles.NeuralModel import train_model, evaluate_tm
from PythonFiles.evaluator import evaluate_audio

c:\Users\mmapa\Desktop\Eti\ASR\ASR_project\.venv\Lib\site-packages\torchaudio\functional\functional.py:581: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(


In [2]:
# 1. Load the entire saved checkpoint into memory
checkpoint = torch.load('8epok_300files.pt', map_location='cpu')

# 2. Initialize your model
model = CRNN()

# 3. Load ONLY the state dictionary from the checkpoint
model.load_state_dict(checkpoint['model_state_dict'])

# 4. Set to evaluation mode
model.eval()

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
evaluate_audio('../data/501-1000/122/100022_997.wav',
                '../data/501-1000/122/100022_997.TextGrid', 
                model=model, device=device, show_per_window=True, top_k=3)


file: ../data/501-1000/122/100022_997.wav
duration: 4.72s, 233 windows

per-window predictions:
 start    end    pred     p   top-3   truth
  0.00   0.08       n  0.70   n:0.70 l:0.14 r:0.05   m(0.01) ✗
  0.02   0.10       a  0.75   a:0.75 sil:0.07 n:0.05   m(0.01) ✗
  0.04   0.12       a  0.94   a:0.94 sil:0.01 e:0.01   m(0.00) ✗
  0.06   0.14       a  0.92   a:0.92 e:0.04 o:0.02   u(0.00) ✗
  0.08   0.16       v  0.24   v:0.24 o:0.24 a:0.18   v(0.24) ✓
  0.10   0.18       v  0.94   v:0.94 f:0.01 i:0.01   v(0.94) ✓
  0.12   0.20       v  0.35   v:0.35 i:0.35 sil:0.18   v(0.35) ✓
  0.14   0.22       i  0.60   i:0.60 e:0.25 sil:0.05   i(0.60) ✓
  0.16   0.24       e  0.73   e:0.73 i:0.11 sil:0.04   i(0.11) ✗
  0.18   0.26       e  0.38   e:0.38 j:0.22 n~:0.21   i(0.02) ✗
  0.20   0.28       j  0.32   j:0.32 e:0.25 n~:0.17   w(0.00) ✗
  0.22   0.30       j  0.26   j:0.26 i:0.22 e:0.21   w(0.00) ✗
  0.24   0.32     sil  0.55   sil:0.55 i:0.22 i2:0.11   tsj(0.00) ✗
  0.26   0.34     sil  

{'pred': ['n',
  'a',
  'a',
  'a',
  'v',
  'v',
  'v',
  'i',
  'e',
  'e',
  'j',
  'j',
  'sil',
  'sil',
  'tsj',
  'tsj',
  'tsj',
  'S',
  'sil',
  'sil',
  'e',
  'h',
  'h',
  'h',
  'h',
  'h',
  'o',
  'o',
  'd',
  'd',
  'd',
  'sil',
  'o',
  'o',
  'o',
  't',
  'k',
  'k',
  'k',
  'o',
  'o',
  'oc5',
  'm',
  'oc5',
  'oc5',
  'g',
  'sil',
  'sil',
  'sil',
  'sil',
  'sil',
  't',
  'o',
  'o',
  'sil',
  'e',
  'e',
  'e',
  'e',
  'e',
  'e',
  'e',
  'e',
  'e',
  'g',
  'sil',
  'k',
  'g',
  'k',
  'k',
  'o',
  'o',
  'n',
  'n',
  'sil',
  'v',
  'sil',
  'tS',
  'p',
  'o',
  'e',
  'e',
  'e',
  'e',
  'e',
  'e',
  'e',
  'e',
  'j',
  'e',
  'e',
  'e',
  'e',
  'e',
  'n',
  'n',
  'sil',
  'g',
  'd',
  't',
  't',
  'i2',
  'o',
  'b',
  'b',
  'b',
  'sil',
  'p',
  'p',
  'e',
  'e',
  'e',
  'e',
  'e',
  'a',
  't',
  'c',
  'tsj',
  'tsj',
  'tsj',
  'tsj',
  'sj',
  'tsj',
  'i',
  'e',
  'j',
  'j',
  'j',
  'i',
  'i',
  'sil',
  'sil',
  'sil'

In [23]:
idx = C.LABEL2IDX
idx

#label = C.IDX2LABEL
#label

{'S': 0,
 'Z': 1,
 'a': 2,
 'b': 3,
 'c': 4,
 'd': 5,
 'dZ': 6,
 'dz': 7,
 'dzj': 8,
 'e': 9,
 'eo5': 10,
 'f': 11,
 'g': 12,
 'h': 13,
 'i': 14,
 'i2': 15,
 'j': 16,
 'k': 17,
 'l': 18,
 'm': 19,
 'n': 20,
 'n~': 21,
 'o': 22,
 'oc5': 23,
 'p': 24,
 'r': 25,
 's': 26,
 'sj': 27,
 'sil': 28,
 'sp': 29,
 't': 30,
 'tS': 31,
 'tsj': 32,
 'u': 33,
 'v': 34,
 'w': 35,
 'z': 36,
 'zj': 37,
 'oov': 38}